# 07e · Önceden kayıtlı test: Seviye + EMA geri çekilme stratejisi
## PDH/PDL · PMH/PML · 15 dk gövde kapanışı · EMA 13/48/200 · 13 EMA'ya geri çekilmede giriş · 1 dk veri, 2016–2026

Bu not defteri **KARARLAR.md → "Önceden kayıtlı test: Seviye + EMA geri çekilme stratejisi (07e)"** bölümündeki kuralları **birebir** uygular. Kurallar 16/11/2026'da, strateji bu veride hiç çalıştırılmadan önce kayda geçirildi (commit `ad8b84a`; stop tanımı gerçek veri görülmeden, sahte veri testinde çıkan sorun üzerine düzeltildi). ORB, 07c–07d sonuçlarından sonra rafa kaldırıldı; referans çıtası 5 dk ORB'nin maliyet sonrası **−0,021 R**'ıdır.

**Akış (belgedeki gibi):**
1. Sabah seviyeleri: PDH, PDL (önceki seans), PMH, PML (04:00–09:29)
2. İlk 30 dakika işlem yok; sinyal mumu en erken 10:00'da başlar
3. 15 dk mum gövdesiyle sinyal: **Kol A** kırılım, **Kol B** ret
4. EMA 13/48/200 sıralaması işlem yönünü desteklemeli
5. Sinyalden sonraki 60 dakikada 5 dk mum 13 EMA'ya değip doğru tarafta kapanırsa kapanışta giriş
6. Stop: kırılan seviye (retde sinyal mumunun ucu) ile giriş mumunun ucundan uzak olanı, en az %0,10 · Hedef: sonraki kilit seviye, yoksa gün sonu

**Doğrulandı (her kol ayrı):** (1) ortalama net R > 0, gün bazında işaret çevirme ile tek yönlü p < **0,0125**, (2) 2016–2020 ve 2021–2026'da ayrı ayrı pozitif, (3) iyimser varsayımda pozitif, (4) en az 200 işlem.

**Veri:** 07c/07d'nin Drive önbelleği (Alpaca seans + piyasa öncesi). Yeni indirme beklenmez; dosya yoksa indirilir. Anahtarlar Colab **Gizli anahtarlar**'dan: `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`.

> ⚠️ Öğrenme amaçlıdır, yatırım tavsiyesi değildir.

In [ ]:
#@title ⚙️ Ayarlar
ONBELLEK = "Google Drive" #@param ["Google Drive", "Colab oturumu"]
PERMUTASYON_TEKRAR = 5000 #@param {type:"integer"}
#@markdown ---
#@markdown **Aşağıdakiler KARARLAR.md'de sabittir — değiştirme.**
TEST_BASLANGIC = "2016-01-01"
TEST_BITIS = "2026-08-21"
ETFLER = ["SPY", "QQQ", "IWM"]
MAG7 = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA"]
KOMISYON_HISSE = 0.0035
KAYMA_BPS = {"ETF": 1, "Mag7": 3}
SON_GIRIS = "15:30"
ILK_SINYAL = "10:00"          # ilk 30 dakika beklenir
GIRIS_PENCERESI_DK = 60
ASGARI_RISK = 0.001           # stop girişten en az %0,10 uzakta
ONCESI_BAS = "04:00"
ONCESI_MIN_MUM = 10
EMA_UZUNLUK = (13, 48, 200)
ALFA = 0.025 / 2               # iki kol (kırılım, ret)
MIN_ISLEM = 200
ORB_REFERANS_R = -0.021        # 07c/07d: 5 dk ORB, maliyet sonrası

## 1 · Kurulum, anahtarlar, önbellek

In [ ]:
import subprocess, sys, os, time, json
try:
    import pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def tr_sayi(x, ond=2):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    s = f"{abs(x):,.{ond}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return ("-" if x < 0 else "") + s

def tr_yuzde(x, ond=1):
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "–"
    return ("-" if x < 0 else "") + "%" + tr_sayi(abs(x) * 100, ond)

def tr_tarih(d):
    return pd.Timestamp(d).strftime("%d/%m/%Y")

def dk(saat):
    s, d = saat.split(":")
    return int(s) * 60 + int(d)

def tr_tablo(t, tam=("işlem", "n", "gün", "işlem (+)", "işlem (−)"), yuzde=("isabet", "pay"), ond=3):
    b = t.copy().astype(object)
    for col in t.columns:
        for idx in t.index:
            x = t.at[idx, col]
            if isinstance(x, str):
                continue
            if col in tam:
                b.at[idx, col] = tr_sayi(x, 0)
            elif any(str(col).startswith(y) for y in yuzde):
                b.at[idx, col] = tr_yuzde(x)
            else:
                b.at[idx, col] = tr_sayi(x, ond)
    return b

from google.colab import userdata
ANAHTAR = {}
for ad in ["ALPACA_API_KEY", "ALPACA_SECRET_KEY"]:
    try:
        ANAHTAR[ad] = userdata.get(ad)
        print(f"✅ {ad} okundu")
    except Exception as e:
        raise RuntimeError(f"{ad} okunamadı. Sol menüdeki anahtar simgesinden ekle ve 'Not defteri erişimi'ni aç. ({e})")

if ONBELLEK == "Google Drive":
    from google.colab import drive
    drive.mount("/content/drive")
    KLASOR = "/content/drive/MyDrive/abd-borsa-calismalari/veri07c"
else:
    KLASOR = "/content/veri07c"
os.makedirs(KLASOR, exist_ok=True)
print("Önbellek klasörü:", KLASOR)

SEMBOLLER = ETFLER + MAG7
GRUP = {s: ("ETF" if s in ETFLER else "Mag7") for s in SEMBOLLER}
ACILIS = dk("09:30")
rng = np.random.default_rng(42)

## 2 · Veri
- **Seans (09:30–16:00):** 07c ile aynı dosyalar (`alpaca_<sembol>`); Drive'da varsa yeniden indirilmez.
- **Piyasa öncesi (04:00–09:30):** yeni dosyalar (`alpaca_oncesi_<sembol>`). Alpaca her gün için tüm uzatılmış seansı döndürür; yalnızca 04:00–09:29 tutulur.
- **META/FB:** META için 2016 başından veri gelmezse FB'den tamamlanır (07c'deki kural).

In [ ]:
ALPACA_URL = "https://data.alpaca.markets/v2/stocks/bars"

def alpaca_indir(sembol, baslangic, bitis, bolum="seans"):
    basliklar = {"APCA-API-KEY-ID": ANAHTAR["ALPACA_API_KEY"], "APCA-API-SECRET-KEY": ANAHTAR["ALPACA_SECRET_KEY"]}
    parametre = {"symbols": sembol, "timeframe": "1Min", "start": f"{baslangic}T00:00:00Z",
                 "end": f"{bitis}T23:59:00Z", "limit": 10000, "adjustment": "raw", "feed": "sip", "sort": "asc"}
    satirlar, sayfa = [], None
    while True:
        if sayfa:
            parametre["page_token"] = sayfa
        for deneme in range(5):
            y = requests.get(ALPACA_URL, headers=basliklar, params=parametre, timeout=60)
            if y.status_code == 429:
                time.sleep(5 * (deneme + 1)); continue
            y.raise_for_status(); break
        j = y.json()
        satirlar.extend(j.get("bars", {}).get(sembol, []) or [])
        sayfa = j.get("next_page_token")
        time.sleep(0.31)                      # ücretsiz plan: dakikada 200 istek
        if not sayfa:
            break
    if not satirlar:
        return pd.DataFrame()
    df = pd.DataFrame(satirlar)[["t", "o", "h", "l", "c", "v"]]
    df.index = pd.to_datetime(df.pop("t"), utc=True).dt.tz_convert("America/New_York")
    dakika = df.index.hour * 60 + df.index.minute
    if bolum == "seans":
        return df[(dakika >= ACILIS) & (dakika < dk("16:00"))].astype("float64")
    return df[(dakika >= dk(ONCESI_BAS)) & (dakika < ACILIS)].astype("float64")

def onbellekten(ad, uret):
    yol = os.path.join(KLASOR, f"{ad}.parquet")
    if os.path.exists(yol):
        return pd.read_parquet(yol)
    df = uret()
    if len(df):
        df.to_parquet(yol)
    return df

def indir_sembol(s, bolum):
    on = "alpaca_" if bolum == "seans" else "alpaca_oncesi_"
    df = onbellekten(f"{on}{s}", lambda: alpaca_indir(s, TEST_BASLANGIC, TEST_BITIS, bolum))
    if s == "META" and (df.empty or df.index.min() > pd.Timestamp("2016-02-01", tz="America/New_York")):
        bitis_fb = (df.index.min() - pd.Timedelta(days=1)).strftime("%Y-%m-%d") if len(df) else TEST_BITIS
        fb = onbellekten(f"{on}FB", lambda: alpaca_indir("FB", TEST_BASLANGIC, bitis_fb, bolum))
        df = pd.concat([fb, df]).sort_index()
        df = df[~df.index.duplicated()]
        print(f"META ({bolum}): eski dönem FB sembolünden tamamlandı.")
    return df

BAR, ONCESI = {}, {}
for s in SEMBOLLER:
    t0 = time.time()
    BAR[s] = indir_sembol(s, "seans")
    ONCESI[s] = indir_sembol(s, "oncesi")
    g1 = pd.Series(BAR[s].index.date).nunique() if len(BAR[s]) else 0
    g2 = pd.Series(ONCESI[s].index.date).nunique() if len(ONCESI[s]) else 0
    print(f"{s:5s}: seans {tr_sayi(g1, 0)} gün · piyasa öncesi {tr_sayi(g2, 0)} gün, "
          f"{tr_sayi(len(ONCESI[s]), 0)} mum ({tr_sayi(time.time() - t0, 0)} sn)")

## 3 · Veri kalite kontrolü
Piyasa öncesi işlem 2016'da ince olabilir (özellikle IWM ve bazı hisselerde). 10'dan az piyasa öncesi mumu olan günler F3 testinden çıkar (kayıttaki kural); aşağıdaki tablo bunun hangi yıllarda ne kadar olduğunu gösterir.

In [ ]:
kalite = {}
for s in SEMBOLLER:
    d = ONCESI[s]
    if d.empty:
        kalite[s] = pd.Series(dtype=float); continue
    say = pd.Series(1, index=d.index).groupby(d.index.date).size()
    yeterli = say[say >= ONCESI_MIN_MUM]
    kalite[s] = pd.Series(1, index=pd.to_datetime(yeterli.index)).groupby(lambda x: x.year).size()
kalite = pd.DataFrame(kalite).fillna(0).astype(int)
display(Markdown(f"**Yıllara göre en az {ONCESI_MIN_MUM} piyasa öncesi mumu olan gün sayısı**"))
display(kalite)

## 4 · Günlere bölme, seviyeler ve EMA
- **PDH/PDL:** aynı sembolün önceki seans gününün (SPY takvimine göre) en yükseği / en düşüğü. Önceki gün verisi yoksa F1 eksik.
- **PMH/PML:** aynı günün 04:00–09:29 mumları.
- **EMA:** seans içi 5 dk mumların kapanışı, günler arası kesintisiz; 13, 48, 200 periyot (`adjust=False`). Girişte yalnızca **tamamlanmış** 5 dk mumlar kullanılır (mumun bitişi ≤ giriş dakikası).

In [ ]:
def gunlere_bol(df):
    gunler = {}
    if df.empty:
        return gunler
    ts = df.index
    tarih = pd.to_datetime(ts.date)
    dakika = (ts.hour * 60 + ts.minute).values
    o, h, l, c = (df[k].values for k in ["o", "h", "l", "c"])
    sinir = np.flatnonzero(np.r_[True, tarih[1:] != tarih[:-1], True])
    for a, b in zip(sinir[:-1], sinir[1:]):
        if dakika[a] != ACILIS:
            continue
        gunler[tarih[a]] = dict(dk=dakika[a:b], o=o[a:b], h=h[a:b], l=l[a:b], c=c[a:b])
    return gunler

GUN = {s: gunlere_bol(BAR[s]) for s in SEMBOLLER}
TAKVIM = sorted(GUN["SPY"].keys())
ONCEKI = {TAKVIM[i]: TAKVIM[i - 1] for i in range(1, len(TAKVIM))}

PM = {}
for s in SEMBOLLER:
    d = ONCESI[s]
    PM[s] = {}
    if d.empty:
        continue
    g = d.groupby(pd.to_datetime(d.index.date))
    t = pd.DataFrame({"h": g["h"].max(), "l": g["l"].min(), "n": g.size()})
    for gun, r in t.iterrows():
        if r["n"] >= ONCESI_MIN_MUM:
            PM[s][gun] = (r["h"], r["l"])

EMA = {}
for s in SEMBOLLER:
    d = BAR[s]
    if d.empty:
        EMA[s] = {}; continue
    b5 = d["c"].resample("5min", label="left", closed="left").last().dropna()
    b5 = b5[(b5.index.hour * 60 + b5.index.minute >= ACILIS) & (b5.index.hour * 60 + b5.index.minute < dk("16:00"))]
    e = pd.DataFrame({f"e{n}": b5.ewm(span=n, adjust=False).mean() for n in EMA_UZUNLUK})
    e["sira"] = np.arange(len(e))
    e["bas"] = e.index.hour * 60 + e.index.minute
    EMA[s] = {pd.Timestamp(gun): x for gun, x in e.groupby(pd.to_datetime(e.index.date))}
print("Günler ayrıldı:", {s: len(GUN[s]) for s in SEMBOLLER})

## 5 · Strateji motoru
- **15 dk mumlar** ve **5 dk mumlar** her günün 1 dk mumlarından kurulur (seans içi).
- **Trend** sinyal mumu kapandığı anda bilinen son tamamlanmış 5 dk mumun EMA'larıdır.
- **Giriş dokunuşu** için bir önceki tamamlanmış 5 dk mumun 13 EMA'sı kullanılır (mum içinde bilinen değer).
- **Çıkış** girişten sonraki 1 dk mumlarla: açılış stopun/hedefin ötesindeyse açılıştan; aynı dakikada ikisi birden mümkünse kötümser = stop, iyimser = hedef.

In [ ]:
def mumlar(g, uzunluk):
    """Seans içi 'uzunluk' dakikalık mumlar: başlangıç dk, açılış, en yüksek, en düşük, kapanış, son 1 dk indeksi."""
    dkk = g["dk"]
    grup = (dkk - ACILIS) // uzunluk
    sinir = np.flatnonzero(np.r_[True, grup[1:] != grup[:-1], True])
    a, b = sinir[:-1], sinir[1:]
    return dict(bas=ACILIS + grup[a] * uzunluk, o=g["o"][a], h=np.maximum.reduceat(g["h"], a),
                l=np.minimum.reduceat(g["l"], a), c=g["c"][b - 1], son=b - 1)

def trend(e, zaman):
    """zaman (dk) itibarıyla tamamlanmış son 5 dk mumunda EMA sıralaması: +1, -1 ya da 0 (yok/karışık)."""
    if e is None:
        return 0
    tamam = e[e["bas"] + 5 <= zaman]
    if tamam.empty:
        return 0
    x = tamam.iloc[-1]
    if x["sira"] < max(EMA_UZUNLUK) - 1:
        return 0
    a, b, c = (x[f"e{n}"] for n in EMA_UZUNLUK)
    return 1 if a > b > c else (-1 if a < b < c else 0)

def cikis(g, bas, stop, hedef, yon):
    """bas: girişten sonraki ilk 1 dk mum indeksi. Döner: (kötümser çıkış, iyimser çıkış, hedefe ulaştı mı)."""
    o, h, l, c = g["o"], g["h"], g["l"], g["c"]
    for i in range(bas, len(o)):
        if yon == 1:
            if o[i] <= stop:
                return o[i], o[i], False
            if hedef is not None and o[i] >= hedef:
                return o[i], o[i], True
            s, t = l[i] <= stop, hedef is not None and h[i] >= hedef
        else:
            if o[i] >= stop:
                return o[i], o[i], False
            if hedef is not None and o[i] <= hedef:
                return o[i], o[i], True
            s, t = h[i] >= stop, hedef is not None and l[i] <= hedef
        if s and t:
            return stop, hedef, None          # belirsiz: kötümser stop, iyimser hedef
        if s:
            return stop, stop, False
        if t:
            return hedef, hedef, True
    return c[-1], c[-1], False

def gun_seviyeleri(s, gun):
    dun = ONCEKI.get(gun)
    g0 = GUN[s].get(dun) if dun is not None else None
    if g0 is None:
        return None
    sev = {"PDH": g0["h"].max(), "PDL": g0["l"].min()}
    pm = PM[s].get(gun)
    if pm is not None:
        sev["PMH"], sev["PML"] = pm
    return sev

def sinyaller(m15, sev, kol):
    """Kronolojik sinyaller: (mum indeksi, yön, seviye adı, stop)."""
    cikti = []
    for j in range(1, len(m15["bas"])):
        if m15["bas"][j] < dk(ILK_SINYAL) or m15["bas"][j] + 15 > dk(SON_GIRIS):
            continue
        o, h, l, c, cp = m15["o"][j], m15["h"][j], m15["l"][j], m15["c"][j], m15["c"][j - 1]
        for ad, L in sev.items():
            ust = ad in ("PDH", "PMH")
            if kol == "A":
                if ust and c > L and cp <= L:
                    cikti.append((j, 1, ad, L))
                elif not ust and c < L and cp >= L:
                    cikti.append((j, -1, ad, L))
            else:
                if ust and o < L and c < L and h >= L:
                    cikti.append((j, -1, ad, h))
                elif not ust and o > L and c > L and l <= L:
                    cikti.append((j, 1, ad, l))
    return cikti

def gun_islemi(s, gun, kol, ema_filtresi=True):
    g = GUN[s][gun]
    sev = gun_seviyeleri(s, gun)
    if sev is None:
        return None
    e = EMA[s].get(gun)
    if e is None or e.empty:
        return None
    m15, m5 = mumlar(g, 15), mumlar(g, 5)
    e13 = pd.Series(e["e13"].values, index=e["bas"].values)
    for j, yon, ad, stop in sinyaller(m15, sev, kol):
        T = m15["bas"][j] + 15
        if ema_filtresi and trend(e, T) != yon:
            continue
        for k in range(len(m5["bas"])):
            b5 = m5["bas"][k]
            if b5 < T or b5 + 5 > T + GIRIS_PENCERESI_DK or b5 + 5 > dk(SON_GIRIS):
                continue
            onceki = b5 - 5
            if onceki not in e13.index:
                continue
            ref = e13[onceki]
            dokundu = (m5["l"][k] <= ref and m5["c"][k] > ref) if yon == 1 else (m5["h"][k] >= ref and m5["c"][k] < ref)
            if not dokundu:
                continue
            giris = m5["c"][k]
            uc = m5["l"][k] if yon == 1 else m5["h"][k]                 # giriş mumunun ucu (13 EMA'nın gerisi)
            stop_g = min(stop, uc) if yon == 1 else max(stop, uc)
            if (giris - stop_g) * yon < ASGARI_RISK * giris:
                stop_g = giris * (1 - ASGARI_RISK * yon)
            risk = (giris - stop_g) * yon
            if risk <= 0:
                break                         # fiyat stopun yanlış tarafında: bu sinyalden işlem yok
            ileride = [v for v in sev.values() if (v - giris) * yon > 0]
            hedef = (min(ileride) if yon == 1 else max(ileride)) if ileride else None
            ck, ci, ulasti = cikis(g, m5["son"][k] + 1, stop_g, hedef, yon)
            return dict(yon=yon, seviye=ad, giris=giris, stop=stop_g, risk=risk, hedef_var=hedef is not None,
                        cikis_k=ck, cikis_i=ci, hedefe=ulasti, giris_dk=int(b5 + 5))
    return None

def net_r(giris, cikis_f, yon, risk, kayma):
    maliyet = 2 * KOMISYON_HISSE + kayma / 1e4 * (giris + cikis_f)
    return ((cikis_f - giris) * yon - maliyet) / risk

def islemler(kol, ema_filtresi=True):
    satir = []
    for s in SEMBOLLER:
        kayma = KAYMA_BPS[GRUP[s]]
        for gun in GUN[s]:
            if gun not in ONCEKI:
                continue
            r = gun_islemi(s, gun, kol, ema_filtresi)
            if r is None:
                continue
            satir.append(dict(sembol=s, grup=GRUP[s], tarih=gun, yon=r["yon"], seviye=r["seviye"],
                              R_k=net_r(r["giris"], r["cikis_k"], r["yon"], r["risk"], kayma),
                              R_i=net_r(r["giris"], r["cikis_i"], r["yon"], r["risk"], kayma),
                              hedef_var=r["hedef_var"], hedefe=r["hedefe"], giris_dk=r["giris_dk"],
                              risk_bps=r["risk"] / r["giris"] * 1e4))
    return pd.DataFrame(satir)

ISLEM = {}
for kol in ["A", "B"]:
    t0 = time.time()
    ISLEM[kol] = islemler(kol)
    d = ISLEM[kol]
    print(f"Kol {kol}: {tr_sayi(len(d), 0)} işlem · {tr_sayi(time.time() - t0, 0)} sn")

## 6 · Önceden kayıtlı test
Her kol için: maliyet sonrası ortalama net R (kötümser) sıfırdan büyük mü? **Gün bazında işaret çevirme:** her günün toplam R'ı rastgele ±1 ile çarpılır (sıfır etkisi altında simetri). Aynı gündeki işlemler birlikte hareket ettiği için gün birimi esastır.

In [ ]:
def isaret_p(d, tekrar):
    gunluk = d.groupby("tarih")["R_k"].sum().values
    n = len(d)
    gozlenen = d["R_k"].mean()
    say, toplam = 0, 0
    while toplam < tekrar:
        isaret = rng.choice([-1.0, 1.0], size=(500, len(gunluk)))
        say += int(((isaret @ gunluk) / n >= gozlenen).sum()); toplam += 500
    return gozlenen, (say + 1) / (toplam + 1)

ADLAR = {"A": "Kol A · kırılım", "B": "Kol B · ret"}
SONUC = []
for kol in ["A", "B"]:
    d = ISLEM[kol]
    if d.empty:
        SONUC.append({"kol": ADLAR[kol], "işlem": 0, "SONUÇ": "DOĞRULANMADI"}); continue
    yil = pd.to_datetime(d["tarih"]).dt.year
    ort, p = isaret_p(d, PERMUTASYON_TEKRAR)
    a, b, i = d.loc[yil <= 2020, "R_k"].mean(), d.loc[yil >= 2021, "R_k"].mean(), d["R_i"].mean()
    sartlar = [p < ALFA, a > 0 and b > 0, i > 0, len(d) >= MIN_ISLEM]
    SONUC.append({"kol": ADLAR[kol], "işlem": len(d), "net R (kötümser)": ort, "p": p,
                  "2016–2020": a, "2021–2026": b, "net R (iyimser)": i,
                  "isabet": (d["R_k"] > 0).mean(), "şartlar": "".join("✅" if x else "❌" for x in sartlar),
                  "SONUÇ": "DOĞRULANDI" if all(sartlar) else "DOĞRULANMADI"})
SONUC = pd.DataFrame(SONUC).set_index("kol")
display(Markdown("### Karar tablosu\nŞartlar sırasıyla: p < 0,0125 · iki dönem pozitif · iyimser pozitif · en az 200 işlem. "
                 f"Referans: 5 dk ORB {tr_sayi(ORB_REFERANS_R, 3)} R."))
display(tr_tablo(SONUC))

## 7 · Raporlanır ama karar vermez

In [ ]:
for kol in ["A", "B"]:
    d = ISLEM[kol]
    if d.empty:
        continue
    display(Markdown(f"**{ADLAR[kol]}**"))
    ozet = pd.DataFrame([{
        "işlem": len(d), "isabet": (d["R_k"] > 0).mean(), "hedefe ulaşma payı": d["hedefe"].eq(True).mean(),
        "pay hedef yok (gün sonu)": 1 - d["hedef_var"].mean(), "ort. risk (bps)": d["risk_bps"].mean(),
        "net R kötümser": d["R_k"].mean(), "net R iyimser": d["R_i"].mean()}], index=["tümü"])
    display(tr_tablo(ozet, yuzde=("isabet", "hedefe", "pay")))

    kir = []
    for ad, alt in ([("PDH/PDL", d[d["seviye"].isin(["PDH", "PDL"])]), ("PMH/PML", d[d["seviye"].isin(["PMH", "PML"])]),
                     ("uzun", d[d["yon"] == 1]), ("kısa", d[d["yon"] == -1]),
                     ("ETF", d[d["grup"] == "ETF"]), ("Mag7", d[d["grup"] == "Mag7"])]
                    + [(s, d[d["sembol"] == s]) for s in SEMBOLLER]):
        kir.append({"kırılım": ad, "işlem": len(alt), "net R kötümser": alt["R_k"].mean(), "isabet": (alt["R_k"] > 0).mean()})
    display(tr_tablo(pd.DataFrame(kir).set_index("kırılım")))

    yil = pd.to_datetime(d["tarih"]).dt.year
    yt = d.groupby(yil).agg(işlem=("R_k", "size"), **{"net R": ("R_k", "mean")}, isabet=("R_k", lambda x: (x > 0).mean()))
    yt.index.name = "yıl"
    display(tr_tablo(yt))

In [ ]:
#@title 🧪 EMA filtresi olmadan (karar vermez)
for kol in ["A", "B"]:
    d0 = islemler(kol, ema_filtresi=False)
    print(f"{ADLAR[kol]} · EMA filtresiz: {tr_sayi(len(d0), 0)} işlem, net R {tr_sayi(d0['R_k'].mean(), 3)} "
          f"(EMA filtreli {tr_sayi(ISLEM[kol]['R_k'].mean(), 3)})")

In [ ]:
#@title 📈 Kümülatif net R (kötümser)
fig, ax = plt.subplots(figsize=(12, 4))
for kol, renk in [("A", "#2471a3"), ("B", "#c0392b")]:
    d = ISLEM[kol]
    if d.empty:
        continue
    k = d.groupby("tarih")["R_k"].sum().cumsum()
    ax.plot(pd.to_datetime(k.index), k.values, label=ADLAR[kol], color=renk, lw=1.2)
ax.axhline(0, color="grey", lw=0.6); ax.set_title("07e · kümülatif net R"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8 · Kayıt ve X gönderisi
Karar satırları KARARLAR.md'deki kayıtlı bölümün altına **değiştirilmeden** eklenecek.

In [ ]:
satirlar = []
for kol, r in SONUC.iterrows():
    satirlar.append(f"{kol}: {r['SONUÇ']} · net R {tr_sayi(r.get('net R (kötümser)'), 3)}, p={tr_sayi(r.get('p'), 4)}, "
                    f"2016–2020 {tr_sayi(r.get('2016–2020'), 3)} / 2021–2026 {tr_sayi(r.get('2021–2026'), 3)}, "
                    f"iyimser {tr_sayi(r.get('net R (iyimser)'), 3)}, isabet {tr_yuzde(r.get('isabet'))}, {tr_sayi(r['işlem'], 0)} işlem")
print("KARARLAR İÇİN SONUÇ SATIRLARI\n" + "-" * 40)
print("\n".join(satirlar))

gonderi = (
    "Önceden kayıtlı test: seviye + EMA geri çekilme stratejisi\n"
    "$SPY $QQQ $IWM + Mag7 · 2016–2026 · 1 dk veri\n\n"
    "PDH/PDL, PMH/PML, 15 dk gövde kapanışı, 13/48/200 EMA, 13 EMA'ya geri çekilmede giriş. Kuralları veriyi görmeden yazdım.\n\n"
    + "\n".join(f"{'Kırılım' if 'A' in kol else 'Ret'}: {'doğrulandı' if r['SONUÇ'] == 'DOĞRULANDI' else 'doğrulanmadı'} "
                f"({tr_sayi(r.get('net R (kötümser)'), 2)} R/işlem)" for kol, r in SONUC.iterrows())
    + "\n\nÖğrenme amaçlıdır, yatırım tavsiyesi değildir."
)
print("\nKOPYALAMAK İÇİN DÜZ METİN\n" + "-" * 40)
print(gonderi)
print("-" * 40, f"\n{len(gonderi)} karakter")